# Sesion 6 MongoDb
Curso de Especialización en Inteligencia Artificial y Big Data
Profesor: Juan Carlos Pérez González
Departamento de Informática – IES de Teis

**Conexión a Servidor MongoDB (contenedor)**


In [1]:
# Instalar pymongo si no está disponible
import subprocess
import sys

try:
    import pymongo
    print(f"pymongo versión {pymongo.__version__} ya está instalado")
except ImportError:
    print("Instalando pymongo...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymongo"])
    print("pymongo instalado correctamente")

pymongo versión 4.15.5 ya está instalado


In [2]:
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

# Conexión a MongoDB (usando localhost y el puerto mapeado)
try:
    client = MongoClient('mongodb://172.18.0.5:27017/')     #dende Compass con localhost e o porto exposto no docker

# Verificar la conexión
    print(client.list_database_names())  # Esto debería mostrar las bases de datos existentes
    print("conexion realizada")
except ConnectionFailure:
    print ("conexión errónea")

conexión errónea


**Creación de una base de datos**

In [3]:
# Conexión al cliente MongoDB
db = client["database"]  # Aquí se crea o selecciona la base de datos

# Seleccionar o crear la colección
coleccion = db["usuarios"]  # La colección "usuarios"

# Documento a insertar
usuario = {
    "nombre": "Maria Sol",
    "email": "maria@example.com",
    "edad": 42
}

# Insertar el documento
resultado = coleccion.insert_one(usuario)
print(f"Documento insertado con ID: {resultado.inserted_id}")

# Verificar que la base de datos y la colección existen
print("Bases de datos en el servidor:", client.list_database_names())
print("Colecciones en la base de datos:", db.list_collection_names())


ServerSelectionTimeoutError: 172.18.0.5:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6967860d3c05eb71afee4f7a, topology_type: Unknown, servers: [<ServerDescription ('172.18.0.5', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('172.18.0.5:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

In [ ]:
usuarios = [
    {"nombre": "Sonia Ruiz", "email": "sonia@example.com", "edad": 28},
    {"nombre": "Luis Blanco", "email": "luis@example.com", "edad": 40},
    {"nombre": "Marta Pérez", "email": "marta@example.com", "edad": 22}
]

# Insertar varios documentos a la vez
resultado = coleccion.insert_many(usuarios)

# Imprimir los IDs generados
print("Documentos insertados con IDs:", resultado.inserted_ids)

**Consultas simples**

In [ ]:
#IMPORTANTE DB SIEMPRE DELANTE
for doc in db.usuarios.find():
    print(doc)

**Consultas por campos**

In [ ]:
# Por un campo

for doc in db.usuarios.find({"edad": {"$gt": 30}}):
    print(doc)

In [ ]:
# Solo algunos campos

for doc in db.usuarios.find({"edad": {"$gt": 30}}, {"nombre": 1, "_id": 0}):
    print(doc)

IMPORTANTE:

$eq → igual a un valor.
    
$ne → distinto de un valor.

$gt → mayor que un valor.

$gte → mayor o igual que un valor.

$lt → menor que un valor.

$lte → menor o igual que un valor.

$in → dentro de una lista de valores.

$nin → no dentro de una lista de valores.

$or → o lógico.

$not → negación.

$nor → “ni… ni…”.

$exists → si un campo existe.    

$type → filtrar por tipo de dato (int, string, bool, etc.).

$size → tamaño de un array.

$all → contiene todos los valores especificados.

$elemMatch → coincidencia de elementos en arrays con condiciones.


In [ ]:
# 1.Usuarios cuyo nombre sea Juan o Ana
print("\nUsuarios Juan o Ana:")
for doc in db.usuarios.find({"nombre": {"$in": ["Juan Carlos", "Ana"]}}):
    print(doc)

# 2. Usuarios que NO tengan email definido
print("\n Usuarios sin email:")
for doc in db.usuarios.find({"email": {"$exists": False}}):
    print(doc)

**Añadir Campos**

In [ ]:
# Añadir un campo 'intereses' vacío a todos los usuarios
db.usuarios.update_many({}, {"$set": {"intereses": []}})

# Verificar los cambios
for doc in db.usuarios.find():
    print(doc)


**Actualizar campos**

In [ ]:
db.usuarios.update_one(
    {"nombre": "Juan Carlos"},
    {"$set": {"intereses": ["Python", "Big Data"]}}
)

# Consultar ese usuario
print(db.usuarios.find_one({"nombre": "Juan Carlos"}))


In [ ]:
#añadir un campo activo
db.usuarios.update_many({}, {"$set": {"activo": True}})

In [ ]:
# Verificar los cambios
for doc in db.usuarios.find():
    print(doc)

**Busquedas Avanzadas**

In [ ]:
# por igualdad

for doc in db.usuarios.find({"edad": 35}):
    print(doc)


In [ ]:
# por comparación
# mayoor qué

for doc in db.usuarios.find({"edad": {"$gt": 40}}):
    print(doc)

#menor que

for doc in db.usuarios.find({"edad": {"$lt": 30}}):
    print(doc)

#mayor y menor o igual

for doc in db.usuarios.find({"edad": {"$gte": 30, "$lte": 40}}):
    print(doc)


In [ ]:
# varios campos como AND o BUSQUEDA COMBINADA

query = {"edad": {"$gt": 30}, "nombre": "Luis Blanco"}
for doc in db.usuarios.find(query):
    print(doc)

In [ ]:
# valores dentro de una lista

for doc in db.usuarios.find({"edad": {"$in": [22, 28]}}):
    print(doc)


In [ ]:
# valores regulares  en este caso que empiecen pr M

for doc in db.usuarios.find({"nombre": {"$regex": "^M", "$options": "i"}}):
    print(doc)


In [ ]:
# seleccionar algunos campos 

for doc in db.usuarios.find({}, {"_id": 0, "nombre": 1, "edad": 1}):
    print(doc)


In [ ]:
#ordenando

for doc in db.usuarios.find().sort("edad", 1):  # 1 ascendente, -1 descendente
    print(doc)


In [ ]:
# limitar resultados
for doc in db.usuarios.find().limit(3):
    print(doc)



**EJERCICIOS**

Antes de avanzar te propongo estos ejercicios:

1. Buscar usuarios mayores de 40.
2. Buscar todos los que empiecen por la letra L.
3. Listar solo nombre y correo, ordenados por edad.
4. Buscar entre edades 30–50.
5. Hacer una búsqueda combinada:
- edad > 30
- nombre que empiece por M



**Actualizaciones avanzadas**

In [ ]:
#empezamos por algo sencillo busca el resultado en COMPAss

db.usuarios.update_one(
    {"nombre": "Juan Carlos"},
    {"$set": {"edad": 36}}
)

print(list(db.usuarios.find({"nombre": "Juan Carlos"})))



In [ ]:
#actuializar varios usuarios subiéndoles un año

db.usuarios.update_many(
    {"edad": {"$gt": 30}},
    {"$inc": {"edad": 1}}
)



In [ ]:
#eliminar un campo

db.usuarios.update_many(
    {},
    {"$unset": {"email": ""}}
)


In [ ]:
#renombrar un campo

db.usuarios.update_many(
    {},
    {"$rename": {"nombre": "fullname"}}
)


In [ ]:
#actualizar un usuarios específico

db.usuarios.update_one(
    {"fullname": "Marta Pérez"},
    {
        "$set": {"edad": 23, "activo": False},
        "$inc": {"puntos": 10}
    }
)


**EJERCICIOS**

1. Añade el campo rol con valor usuario a todos los documentos de la colección usuarios
2. Añadel el campo puntos con +20 al usuario Luis Angel
3. Añade el campo email a Marta Pérez con marta@mail.com y un campo premium: true
4. Añade al mismo usuario Marta Pérez un campo intereses con JavaScript y Docker

**Eliminaciones**

In [ ]:
# Borra un usuario específico por nombre
resultado = coleccion.delete_one({"fullname": "Luis Blanco"})
print(f"Documentos eliminados: {resultado.deleted_count}")


In [ ]:
# Borra todos los usuarios que no estén activos
resultado = coleccion.delete_many({"activo": False})
print(f"Documentos eliminados: {resultado.deleted_count}")


In [ ]:
# Vaciar la colección
resultado = coleccion.delete_many({})
print(f"Documentos eliminados: {resultado.deleted_count}")


**Índices**

Los índices son estructuras de datos que mejoran el rendimiento de las consultas. Sin índices, MongoDB debe escanear todos los documentos (collection scan). Con índices, accede directamente a los documentos que coinciden.

In [ ]:
# Crear un índice simple en el campo 'edad'
db.usuarios.create_index([("edad", 1)])  # 1 = ascendente, -1 = descendente
print("Índice en 'edad' creado")

# Listar todos los índices en la colección
indices = db.usuarios.list_indexes()
for indice in indices:
    print(indice)

In [ ]:
# Índice compuesto (múltiples campos)
db.usuarios.create_index([("edad", 1), ("fullname", 1)])
print("Índice compuesto en 'edad' y 'fullname' creado")

In [ ]:
# Índice único (no permite valores duplicados)
db.usuarios.create_index([("email", 1)], unique=True)
print("Índice único en 'email' creado")

# Intentar insertar un email duplicado (generará error)
try:
    db.usuarios.insert_one({"fullname": "Nuevo Usuario", "email": "maria@example.com"})
except Exception as e:
    print(f"Error al insertar duplicado: {e}")

In [ ]:
# Índice de texto (búsqueda por palabras)
db.usuarios.create_index([("fullname", "text"), ("email", "text")])
print("Índice de texto creado")

# Buscar documentos con texto
for doc in db.usuarios.find({"$text": {"$search": "Maria"}}):
    print(doc)

**Información sobre índices y borrado**

In [ ]:
# Borrar un índice específico
db.usuarios.drop_index([("edad", 1)])
print("Índice en 'edad' eliminado")

# Borrar todos los índices (excepto el _id que es obligatorio)
db.usuarios.drop_indexes()
print("Todos los índices eliminados (excepto _id)")

**Tipos de Índices - Resumen**

| Tipo | Uso | Sintaxis |
|------|-----|---------|
| **Simple** | Campo único | `create_index([("campo", 1)])` |
| **Compuesto** | Múltiples campos | `create_index([("campo1", 1), ("campo2", 1)])` |
| **Único** | Sin duplicados | `create_index([("campo", 1)], unique=True)` |
| **Texto** | Búsqueda por palabras | `create_index([("campo", "text")])` |
| **TTL** | Expiración automática | `create_index([("fecha", 1)], expireAfterSeconds=3600)` |
| **Geoespacial** | Coordenadas | `create_index([("ubicacion", "2dsphere")])` |

**EJERCICIOS - Índices**

1. Crea un índice en el campo `edad` y lista todos los índices.
2. Crea un índice compuesto en `edad` y `fullname`.
3. Crea un índice único en `email` e intenta insertar un email duplicado.
4. Crea un índice de texto en `fullname` y busca por palabra.
5. Borra todos los índices y verifica que solo queda el índice `_id`.

**CARGA DE DATOS DESDE FICHEROS**

En este caso hablaremos de csv

In [ ]:
import csv
from pymongo import MongoClient
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

# Conexión a MongoDB (usando la IP interna del contenedor: 172.18.0.3)
try:
    # Corrección: Usar la IP 172.18.0.3 que se ve en la inspección de red de Docker
    client = MongoClient('mongodb://172.18.0.5:27017/')

    print("conexion realizada")

    db = client["database"]        # Base de datos
    coleccion = db["empleados"]   # Colección

    # -------------------------------
    # 2. Leer el CSV
    # -------------------------------
    empleados = []
    with open("empresa.csv", "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Convertir tipos (CSV son strings)
            row["id"] = int(row["id"])
            row["salario"] = int(row["salario"])
            row["activo"] = True if row["activo"] == "True" else False

            empleados.append(row)

    # -------------------------------
    # 3. Insertar en MongoDB
    # -------------------------------
    resultado = coleccion.insert_many(empleados)

    print(f"Documentos insertados: {len(resultado.inserted_ids)}")

except ConnectionFailure:
    print ("conexión errónea")

**Ejercicios**

**A. Consultas Básicas**

1. Obtén todos los empleados del departamento "Ventas".
2. Muestra únicamente los campos nombre y salario de todos los empleados.
3. Consulta los empleados cuyo salario sea mayor de 30000.
4. Filtra los empleados cuya edad esté entre 30 y 40 años, ambos inclusive.
5. Saca todos los empleados inactivos (activo = false).
6. Muestra los empleados cuyo nombre empiece por la letra A.

In [ ]:
coleccion = db["empleados"]

In [ ]:
print("1. Empleados del departamento 'Ventas':")
query_ventas = {"departamento": "Ventas"}
for empleado in coleccion.find(query_ventas).limit(20):
    print(empleado)

In [ ]:
print("\n2. Nombre y Salario de todos los empleados:")
projection_nombre_salario = {"nombre": 1, "salario": 1, "_id": 0}
for empleado in coleccion.find({}, projection_nombre_salario).limit(20):
    print(empleado)

In [ ]:
print("\n3. Empleados con salario mayor de 30000:")
query_salario_mayor = {"salario": {"$gt": 30000}}
for empleado in coleccion.find(query_salario_mayor).limit(20):
    print(empleado)

In [ ]:
print("\n4. Empleados con edad entre 30 y 40 años (Sintaxis correcta para campo 'edad'):")
# Recordatorio: Este query solo funcionará si los documentos tienen un campo 'edad'
query_edad_rango = {"edad": {"$gte": 30, "$lte": 40}}
for empleado in coleccion.find(query_edad_rango).limit(20):
    print(empleado)

In [ ]:
print("\n5. Empleados inactivos (activo = False):")
query_inactivos = {"activo": False}
for empleado in coleccion.find(query_inactivos).limit(20):
    print(empleado)

In [ ]:
print("\n6. Empleados cuyo nombre empieza por 'A' (Insensible a mayúsculas/minúsculas):")
query_nombre_empieza_a = {"nombre": {"$regex": "^A", "$options": "i"}}
for empleado in coleccion.find(query_nombre_empieza_a).limit(20):
    print(empleado)

**B. Consultas Avanzadas**

1. Obtén los empleados cuyo salario no esté entre 20000 y 40000.
2. Busca los empleados cuyo departamento esté en una lista (por ejemplo: Ventas, IT, RRHH).
3. Encuentra empleados con salario superior a la media (primero obtén la media con aggregate).
4. Muestra empleados cuyo nombre contenga "ez" (búsqueda regex).

In [ ]:
#las llaves son el and 
for doc in db.empleados.find({"salario":{"$gte":40000}, {"$lte":20000}}),
{"nombre":1, "salario"}
print("1. Empleados con salario NO entre 20000 y 40000:")
query_salario_fuera_rango = {
    "$or": [
        {"salario": {"$lt": 20000}},
        {"salario": {"$gt": 40000}}
    ]
}

for empleado in coleccion.find(query_salario_fuera_rango).limit(20):
    print(f"{empleado['nombre']} {empleado['apellido']} - Salario: {empleado['salario']}")

In [ ]:
print("\n2. Empleados en Ventas, IT o RRHH:")
lista_departamentos = ["Ventas", "IT", "RRHH"]
query_departamento_in = {"departamento": {"$in": lista_departamentos}}

for empleado in coleccion.find(query_departamento_in).limit(10):
    print(f"{empleado['nombre']} - Departamento: {empleado['departamento']}")

In [ ]:
print("\n3. Empleados con salario superior a la media:")

# --- Paso 1: Calcular el salario medio usando Aggregation Pipeline ---
pipeline_media = [
    {"$group": {"_id": None, "salario_medio": {"$avg": "$salario"}}}
]
resultado_media = list(coleccion.aggregate(pipeline_media))

if resultado_media:
    salario_medio = resultado_media[0]["salario_medio"]
    print(f"-> Salario Medio de la empresa: {salario_medio:.2f}")

    # --- Paso 2: Consultar empleados con salario > media ---
    query_salario_superior_media = {"salario": {"$gt": salario_medio}}
    
    for empleado in coleccion.find(query_salario_superior_media).sort("salario", -1).limit(10): # Ordenado por salario descendente
        print(f"{empleado['nombre']} {empleado['apellido']} - Salario: {empleado['salario']}")
else:
    print("No se encontraron datos para calcular la media.")

In [ ]:
print("\n4. Empleados cuyo nombre contenga 'ez':")
# Expresión regular: /ez/i
query_nombre_contiene_ez = {"nombre": {"$regex": "ez", "$options": "i"}}

for empleado in coleccion.find(query_nombre_contiene_ez):
    print(f"{empleado['nombre']} {empleado['apellido']} - Email: {empleado['email']}")

**C. Ordenación y Límites**

1. Lista los 10 empleados con mayor salario.
2. Devuelve los empleados ordenados por edad descendente.

In [ ]:
print("1. Los 10 empleados con mayor salario:")

# Pipeline: Ordenar por salario descendente (-1) y limitar a 10
results = coleccion.find({}) \
    .sort("salario", -1) \
    .limit(10)

# Mostrar resultados con los campos relevantes
for empleado in results:
    print(f"Salario: {empleado['salario']}, Nombre: {empleado['nombre']} {empleado['apellido']}, Depto: {empleado['departamento']}")

In [ ]:
print("\n2. Empleados ordenados por edad descendente (Sintaxis correcta para campo 'edad'):")

# Pipeline: Ordenar por edad descendente (-1)
# Advertencia: Esta consulta puede no devolver resultados ya que 'edad' no existe en empresa.csv
results = coleccion.find({}).limit(10) \
    .sort("edad", -1)

# Mostrar resultados
for empleado in results:
    # Se añade una verificación simple para evitar errores si el campo 'edad' no existe
    edad_display = empleado.get('edad', 'N/D')
    print(f"Edad: {edad_display}, Nombre: {empleado['nombre']} {empleado['apellido']}")

**D. Agregaciones**
1. Calcula el salario medio por departamento.
2. Cuenta cuántos empleados hay por cada departamento.
3. Devuelve el empleado mejor pagado de cada departamento.
4. Suma el salario total de toda la empresa.
5. Consulta qué departamento tiene más empleados activos.
6. Obtén una lista con los distintos departamentos existentes ($group).

In [ ]:
print("1. Salario Medio por Departamento:")

salario_medio = [
    {
        "$group": {
            "_id": "$departamento",
            "salario_medio": {"$avg": "$salario"}
        }
    },
    # Opcional: ordenar por salario medio descendente
    {"$sort": {"salario_medio": -1}}
]

for resultado in coleccion.aggregate(salario_medio):
    print(f"Departamento: {resultado['_id']}, Salario Medio: {resultado['salario_medio']:.2f}")

In [ ]:
print("\n2. Conteo de Empleados por Departamento:")

pipeline_conteo = [
    {
        "$group": {
            "_id": "$departamento",
            "total_empleados": {"$sum": 1}
        }
    },
    # Opcional: ordenar por el total de empleados descendente
    {"$sort": {"total_empleados": -1}}
]

for resultado in coleccion.aggregate(pipeline_conteo):
    print(f"Departamento: {resultado['_id']}, Total Empleados: {resultado['total_empleados']}")

In [ ]:
print("\n3. Empleado Mejor Pagado de cada Departamento:")

pipeline_mejor_pagado = [
    # 1. Ordenar por departamento (para agrupar) y luego por salario descendente
    {"$sort": {"departamento": 1, "salario": -1}},
    
    # 2. Agrupar por departamento y tomar el primer documento (que es el de mayor salario)
    {
        "$group": {
            "_id": "$departamento",
            "max_salario": {"$first": "$salario"},
            "nombre": {"$first": "$nombre"},
            "apellido": {"$first": "$apellido"},
        }
    },
    # 3. Opcional: ordenar por el salario máximo
    {"$sort": {"max_salario": -1}}
]

for resultado in coleccion.aggregate(pipeline_mejor_pagado):
    print(f"Depto: {resultado['_id']}, Nombre: {resultado['nombre']} {resultado['apellido']}, Salario: {resultado['max_salario']}")

In [ ]:
print("\n4. Suma del Salario Total de toda la Empresa:")

pipeline_salario_total = [
    {
        "$group": {
            "_id": None,
            "salario_total": {"$sum": "$salario"}
        }
    }
]

resultado = list(coleccion.aggregate(pipeline_salario_total))

if resultado:
    print(f"Salario Total: {resultado[0]['salario_total']:.2f}")

In [ ]:
print("\n5. Departamento con más Empleados Activos:")

pipeline_activos = [
    # 1. Filtrar solo los empleados activos
    {"$match": {"activo": True}},
    
    # 2. Contar los empleados activos por departamento
    {"$group": {"_id": "$departamento", "empleados_activos": {"$sum": 1}}},
    
    # 3. Ordenar para poner el máximo primero
    {"$sort": {"empleados_activos": -1}},
    
    # 4. Limitar al primer resultado (el que tiene el mayor conteo)
    {"$limit": 1}
]

resultado = list(coleccion.aggregate(pipeline_activos))

if resultado:
    print(f"El departamento con más activos es: {resultado[0]['_id']} con {resultado[0]['empleados_activos']} empleados.")

In [ ]:
print("\n6. Lista de Departamentos Únicos:")

pipeline_departamentos = [
    {
        "$group": {
            "_id": "$departamento"
        }
    }
]

departamentos = [doc['_id'] for doc in coleccion.aggregate(pipeline_departamentos)]

print(departamentos)

**E. Actualizaciones**

1. Incrementa el salario un 5% a todos los empleados del departamento IT.
2. Marca como inactivos todos los empleados cuyo salario sea inferior a 18000.

In [ ]:
print("1. Incrementando salario un 5% a empleados de IT...")

# 1. Definir la consulta (query) para encontrar a los empleados de IT
query_it = {"departamento": "IT"}

# 2. Definir la actualización (update) usando el operador $mul
# $mul: Multiplica el valor actual de 'salario' por 1.05
update_incremento = {"$mul": {"salario": 1.05}}

# 3. Ejecutar la actualización masiva
resultado = coleccion.update_many(query_it, update_incremento)

print(f"Documentos coincidentes: {resultado.matched_count}")
print(f"Documentos modificados: {resultado.modified_count}")

# Opcional: Verificar un empleado de IT
print("\nVerificación (Ejemplo de empleado de IT):")
for empleado in coleccion.find(query_it).limit(2):
    print(f"Nombre: {empleado['nombre']} {empleado['apellido']}, Nuevo Salario: {empleado['salario']:.2f}")

In [ ]:
print("\n2. Marcando como inactivos a empleados con salario < 18000...")

# 1. Definir la consulta (query) para encontrar salarios menores a 18000
query_salario_bajo = {"salario": {"$lt": 18000}}

# 2. Definir la actualización (update) usando el operador $set
# $set: Establece el valor del campo 'activo' a False
update_inactivo = {"$set": {"activo": False}}

# 3. Ejecutar la actualización masiva
resultado = coleccion.update_many(query_salario_bajo, update_inactivo)

print(f"Documentos coincidentes: {resultado.matched_count}")
print(f"Documentos modificados: {resultado.modified_count}")

# Opcional: Verificar los empleados inactivos recién marcados
print("\nVerificación (Ejemplos de inactivos con salario bajo):")
for empleado in coleccion.find({"salario": {"$lt": 18000}}).limit(3):
    print(f"Nombre: {empleado['nombre']} {empleado['apellido']}, Salario: {empleado['salario']:.2f}, Activo: {empleado['activo']}")

**F. Eliminaciones**

1. Elimina un empleado por email concreto.
2. Borra todos los empleados inactivos.
3. Elimina empleados con salario inferior a 30000.
4. Elimina empleados de un departamento específico.
5. Vacía la colección empleados.

In [ ]:
print("1. Eliminando empleado por email concreto (elena.martínez1@empresa.com)...")

# Email de ejemplo a eliminar
email_a_eliminar = "elena.martínez1@empresa.com"

query_email = {"email": email_a_eliminar}

# Ejecutar la eliminación de un solo documento
resultado = coleccion.delete_one(query_email)

print(f"Documentos eliminados: {resultado.deleted_count}")

# Verificar si el documento aún existe
print(f"Verificación: ¿Existe {email_a_eliminar}? -> {coleccion.find_one(query_email) is not None}")

In [ ]:
# [Recordatorio: Si ejecutaste el punto 1, considera recargar los datos antes de ejecutar este punto]
print("\n2. Borrando todos los empleados inactivos (activo = False)...")

query_inactivos = {"activo": False}

# Ejecutar la eliminación masiva
resultado = coleccion.delete_many(query_inactivos)

print(f"Documentos inactivos eliminados: {resultado.deleted_count}")

# Verificar cuántos quedan inactivos
print(f"Verificación: Total de documentos inactivos restantes: {coleccion.count_documents(query_inactivos)}")

In [ ]:
# [Recordatorio: Si ejecutaste los puntos anteriores, considera recargar los datos]
print("\n3. Eliminando empleados con salario inferior a 30000...")

query_salario_bajo = {"salario": {"$lt": 30000}}

# Ejecutar la eliminación masiva
resultado = coleccion.delete_many(query_salario_bajo)

print(f"Documentos eliminados por salario bajo: {resultado.deleted_count}")

# Verificar cuántos quedan con salario bajo
print(f"Verificación: Total de documentos con salario < 30000 restantes: {coleccion.count_documents(query_salario_bajo)}")

In [ ]:
# [Recordatorio: Si ejecutaste los puntos anteriores, considera recargar los datos]
print("\n4. Eliminando empleados del departamento 'Logística'...")

departamento_a_eliminar = "Logística"
query_departamento = {"departamento": departamento_a_eliminar}

# Ejecutar la eliminación masiva
resultado = coleccion.delete_many(query_departamento)

print(f"Documentos del departamento '{departamento_a_eliminar}' eliminados: {resultado.deleted_count}")

# Verificar cuántos quedan en ese departamento
print(f"Verificación: Total de documentos en '{departamento_a_eliminar}' restantes: {coleccion.count_documents(query_departamento)}")

In [ ]:
# [Recordatorio: Esta acción elimina TODOS los documentos]
print("\n5. Vaciando la colección de empleados...")

# El query vacío {} coincide con todos los documentos
query_todo = {}

# Ejecutar la eliminación masiva de toda la colección
resultado = coleccion.delete_many(query_todo)

print(f"Documentos eliminados (Colección vaciada): {resultado.deleted_count}")

# Verificar el conteo total
print(f"Verificación: Total de documentos en la colección: {coleccion.count_documents({})}")

G. **Indices**

1. Vuelve a cargar el fichero empresa.csv
2. Crea un índice sobre el campo nombre y comprueba cómo cambia el tiempo de consulta.
3. Crea un índice compuesto por departamento + salario e identifica para qué tipo de consultas es útil.
4. Comprueba en Compass qué índices existen actualmente en la colección.
5. Realiza una consulta que se beneficie de un índice y observa el plan de ejecución (explain()).
6. Elimina un índice y comprueba cómo afecta al rendimiento.


In [ ]:
import csv
from pymongo.errors import ConnectionFailure

print("1. Recargando el fichero empresa.csv en la colección 'empleados'...")

try:
    # Asegurarse de que la conexión esté activa (usando la IP de Docker corregida)
    # Si la conexión está en una celda anterior, no es necesario redefinir client aquí
    # client = MongoClient('mongodb://172.18.0.3:27017/') 
    
    db = client["database"]
    coleccion = db["empleados"]
    
    empleados = []
    with open("empresa.csv", "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Conversión de tipos
            row["id"] = int(row["id"])
            row["salario"] = int(row["salario"])
            row["activo"] = True if row["activo"] == "True" else False
            empleados.append(row)
    
    # Limpiar y reinsertar
    coleccion.delete_many({})
    coleccion.insert_many(empleados)
    print(f"-> Datos recargados. Total de documentos: {coleccion.count_documents({})}")

except ConnectionFailure:
    print("Error: Conexión a MongoDB fallida. Asegúrate de que el contenedor esté corriendo.")
except FileNotFoundError:
    print("Error: No se encontró el archivo 'empresa.csv'.")

In [ ]:
import time

campo_indice_simple = "nombre"

# --- Paso 2A: Medir tiempo sin índice ---
print("\n2A. Consulta sin índice en el campo 'nombre'...")
tiempo_inicio_sin = time.time()
coleccion.find({campo_indice_simple: "Ana"}).explain() # Usamos explain() para no cargar todos los datos, pero simular la búsqueda
tiempo_fin_sin = time.time()
print(f"Tiempo sin índice: {(tiempo_fin_sin - tiempo_inicio_sin) * 1000:.2f} ms")

# --- Paso 2B: Crear índice ---
coleccion.create_index([(campo_indice_simple, 1)])
print(f"-> Índice simple en '{campo_indice_simple}' creado.")

# --- Paso 2C: Medir tiempo con índice ---
print("2C. Consulta con índice en el campo 'nombre'...")
tiempo_inicio_con = time.time()
coleccion.find({campo_indice_simple: "Ana"}).explain()
tiempo_fin_con = time.time()
print(f"Tiempo CON índice: {(tiempo_fin_con - tiempo_inicio_con) * 1000:.2f} ms")

In [ ]:
print("\n3. Creando índice compuesto en 'departamento' y 'salario'...")

coleccion.create_index([("departamento", 1), ("salario", -1)])
print("-> Índice compuesto en (departamento, salario) creado.")

print("\nUtilidad del Índice Compuesto (departamento, salario):")
print("* Consultas que filtran por DEPARTAMENTO (ej. `departamento='IT'`).")
print("* Consultas que filtran por DEPARTAMENTO y ordenan por SALARIO (ej. `departamento='IT'`.sort(`salario`, -1)).")
print("* Consultas que filtran solo por DEPARTAMENTO y/o SALARIO.")

In [ ]:
print("\n4. Índices existentes en la colección 'empleados':")
indices = coleccion.list_indexes()
for indice in indices:
    print(indice)

print("\n-> Verificación manual en Compass: Conéctate a tu servidor MongoDB, selecciona la base de datos 'database', la colección 'empleados' y ve a la pestaña 'Indexes'.")

In [ ]:
print("\n5. Consulta que se beneficia del índice (departamento, salario) y plan de ejecución:")

# Consulta: Buscar empleados en 'IT' con salario > 60000
consulta_beneficiada = {"departamento": "IT", "salario": {"$gt": 60000}}

# Obtener el plan de ejecución
plan_ejecucion = coleccion.find(consulta_beneficiada).explain()

print(f"Resultados de la consulta: {coleccion.count_documents(consulta_beneficiada)}")
print("\n--- Plan de Ejecución (Winning Plan) ---")
print(f"Tipo de Scan: {plan_ejecucion['queryPlanner']['winningPlan']['stage']}")
print(f"Índice Utilizado: {plan_ejecucion['queryPlanner']['winningPlan'].get('inputStage', {}).get('indexName', 'N/A')}")

# Si el índice se utiliza, el 'stage' debe ser IXSCAN o similar.

In [ ]:
print("\n6. Eliminando el índice 'nombre_1' y comprobando el rendimiento...")

# El nombre del índice por defecto es {campo}_{orden}, es decir, 'nombre_1'
nombre_indice_a_eliminar = "nombre_1"
coleccion.drop_index(nombre_indice_a_eliminar)
print(f"-> Índice '{nombre_indice_a_eliminar}' eliminado.")

# --- Medir tiempo después de eliminar el índice ---
tiempo_inicio_post_delete = time.time()
coleccion.find({campo_indice_simple: "Ana"}).explain()
tiempo_fin_post_delete = time.time()
print(f"Tiempo POST-ELIMINACIÓN: {(tiempo_fin_post_delete - tiempo_inicio_post_delete) * 1000:.2f} ms")

print("\nÍndices restantes:")
for indice in coleccion.list_indexes():
    print(indice)